# Conflict against conventional outlier detection

Reviewer 2, comment 3: *if I already use outlier detection, what does pypasi add?* This notebook
compares conflict with the outlier statistics that Raman workflows typically use. Each detector
is fitted on the same training spectra as the band models (10 isolates, SNV):

- **Hotelling's T²** and the **Q residual** of a 10-component PCA model, the score-space and
  residual diagnostics of conventional chemometrics;
- an **isolation forest** and a **local outlier factor** (scikit-learn), two generic detectors;
- pypasi's own `NoveltyDetector`, which combines T² and Q against calibrated limits.

The detectors are tested on two tasks, over five patient-disjoint splits:

1. the planted defects of the paper (five held-out isolates, half of them degraded);
2. the real 2018 → 2019 instrument change (held-out 2018 isolates against the 2019 cohort).

Two questions are asked. First, how well does each score separate degraded spectra on its own?
Second, what does conflict add to a conventional detector combining T², Q and the classifier's
confidence? The second is measured out of sample: weights are fitted on one half of the spectra
and the AUROC is read on the other.

**Requirements.** `pypasi >= 0.5.5` and the Ho et al. `.npy` files in `Bacteria/`. Runtime is about 8 minutes.

## 0 · Setup (shared with the other revision notebooks)

In [1]:
# --- shared setup: the data, the patient-disjoint split and the defects of the paper ---
from pathlib import Path
import time, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
import pypasi
from pypasi import BandNegotiationClassifier, BandSet, ControlProfile, PLSDA, Triage, degrade
from pypasi.datasets import load_bacteria

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Control limits built from a random split")
assert tuple(int(p) for p in pypasi.__version__.split(".")[:3]) >= (0, 5, 5), "needs pypasi >= 0.5.5"

DATA_DIR = Path("Bacteria")
SEED = 0
# the configuration of softwarex_examples.ipynb; every function below defaults to it
N_BANDS, TOPOLOGY, REGIME, N_COMPONENTS, SCALE_NAME = 7, "complete", "H1", 10, "SNV"
CALIBRATION, CAL_SPLIT = "auto", 0.25
REVIEW_RATE, NOVELTY_Q = 0.10, 0.99
K_SIGMA, LASER_NM, PROFILE_STATS = 3.0, 633.0, ("mean_stress", "band_confidence")
BATCH_N, LOCALISE_REPEATS = 100, 6
SPECTRA_PER_PATIENT = 400


def minmax(X):
    X = np.asarray(X, float)
    lo, hi = X.min(1, keepdims=True), X.max(1, keepdims=True)
    return (X - lo) / np.where(hi > lo, hi - lo, 1.0)


def snv(X):
    X = np.asarray(X, float)
    s = X.std(1, keepdims=True)
    return (X - X.mean(1, keepdims=True)) / np.where(s > 0, s, 1.0)


SCALERS = {"SNV": snv, "min-max": lambda X: np.asarray(X, float)}

data = load_bacteria(DATA_DIR, normalize=False)
AXIS = data.axis
MM_all, y_all = minmax(data.X_dev), data.y_dev      # 2018 cohort, per-spectrum min-max
MM_B, y_B = minmax(data.X_test), data.y_test        # 2019 cohort (external)

# Patient = block of 400 consecutive spectra; verified, never assumed
groups = np.arange(len(y_all)) // SPECTRA_PER_PATIENT
for g in np.unique(groups):
    assert np.unique(y_all[groups == g]).size == 1, "row order is not grouped by patient"
N_PAT = groups.max() + 1
PAT_CLASS = np.array([y_all[groups == g][0] for g in range(N_PAT)])

BAND_RANGE = (400.0, min(1800.0, float(AXIS.max())))
REF_BANDS = BandSet.equal_width(AXIS, 7, lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
REF_EDGES = [(b.lo, b.hi) for b in REF_BANDS.bands]   # where local defects are planted


def _take(pool, per_class, rng):
    taken = []
    for c in np.unique(PAT_CLASS[pool]):
        taken.extend(rng.choice(pool[PAT_CLASS[pool] == c], per_class, replace=False))
    taken = np.array(sorted(taken))
    return taken, np.setdiff1d(pool, taken)


def make_split(split_repeat=0):
    """The paper's split (fit 10 | limits 5 | select 5 | qc 5 isolates) and its defects."""
    rng = np.random.default_rng(SEED + split_repeat)
    pool = np.arange(N_PAT)
    p_qc, pool = _take(pool, 1, rng)
    p_sel, pool = _take(pool, 1, rng)
    p_lim, pool = _take(pool, 1, rng)
    rows = lambda p: np.flatnonzero(np.isin(groups, p))  # noqa: E731
    ix = {"fit": rows(pool), "lim": rows(p_lim), "sel": rows(p_sel), "qc": rows(p_qc)}
    ix["ref"] = np.sort(np.concatenate([ix["lim"], ix["sel"], ix["qc"]]))
    return {
        "repeat": split_repeat, "p_qc": p_qc, "ix": ix,
        "chart_patients": np.sort(np.concatenate([p_lim, p_sel])),
        "sel_def": degrade.plant(MM_all[ix["sel"]], AXIS, edges=REF_EDGES, random_state=SEED + 77),
        "qc_def": degrade.plant(MM_all[ix["qc"]], AXIS, edges=REF_EDGES, random_state=SEED),
    }


def binf(X, f):
    """Average f adjacent channels: a coarser detector sampling."""
    X = np.asarray(X, float)
    if f == 1:
        return X
    n = (X.shape[-1] // f) * f
    return X[..., :n].reshape(*X.shape[:-1], n // f, f).mean(-1)


def fit_model(sp, *, n_bands=N_BANDS, topology=TOPOLOGY, scale=SCALE_NAME, mode="equal",
              regime=REGIME, n_components=N_COMPONENTS, bin_factor=1):
    """Fit the band ensemble and its Triage exactly as the paper does, for one configuration."""
    S = SCALERS[scale]
    prep = lambda X: S(binf(X, bin_factor))  # noqa: E731
    ax = binf(AXIS, bin_factor)
    ix = sp["ix"]
    if mode == "equal":
        bands = BandSet.equal_width(ax, n_bands, lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
    else:
        bands = BandSet.peak_informed(ax, binf(MM_all[ix["fit"]], bin_factor), n_bands,
                                      lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
    lim = (prep(MM_all[ix["lim"]]), y_all[ix["lim"]])
    t0 = time.perf_counter()
    model = BandNegotiationClassifier(
        bands=bands, base_estimator=PLSDA(n_components=n_components), topology=topology,
        regime=regime, calibration=CALIBRATION, calibration_data=lim,
        calibration_split=CAL_SPLIT, random_state=SEED).fit(prep(MM_all[ix["fit"]]), y_all[ix["fit"]])
    t_fit = time.perf_counter() - t0
    tri = Triage(conflict_score="dg", review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,
                 n_components=n_components).fit(
        model, *lim, orientation_data=(prep(sp["sel_def"].X), sp["sel_def"].degraded))
    return {"model": model, "triage": tri, "bands": bands, "prep": prep, "t_fit": t_fit}


def localise(sp, fm, planted_intervals=None, trials=None):
    """Plant a local defect in a batch of 100 spectra of one qc isolate; does the chart name it?

    A hit is a named band that overlaps the planted interval; chance is the share of bands
    that overlap it. By default the seven published intervals, six trials each (42 trials).
    """
    prep, bands, model = fm["prep"], fm["bands"], fm["model"]
    edges, labs = [(b.lo, b.hi) for b in bands.bands], bands.labels
    rngb = np.random.default_rng(SEED + 11)
    ref = [prep(MM_all[np.sort(rngb.choice(np.flatnonzero(groups == g), BATCH_N, replace=False))])
           for g in sp["chart_patients"]]
    prof = ControlProfile.fit(model, batches=ref, k=K_SIGMA, laser_nm=LASER_NM,
                              statistics=PROFILE_STATS, random_state=SEED)
    if trials is None:
        trials = [(lo, hi, np.random.default_rng(500 * t + rep))
                  for t, (lo, hi) in enumerate(REF_EDGES) for rep in range(LOCALISE_REPEATS)]
    hits, chance = [], []
    for lo, hi, r in trials:
        sel = (AXIS >= lo) & (AXIS < hi)
        overlap = [j for j, (a, b) in enumerate(edges) if a < hi and b > lo]
        g = r.choice(sp["p_qc"])
        Xl = MM_all[np.sort(r.choice(np.flatnonzero(groups == g), BATCH_N, replace=False))].copy()
        Xl[:, sel] = Xl[:, sel] * 0.55 + r.normal(0, 0.02, (len(Xl), int(sel.sum())))
        tab = prof.check(prep(Xl), name="trial").table
        named = tab.assign(az=tab["z"].abs()).groupby("label")["az"].max().idxmax()
        hits.append(labs.index(named) in overlap)
        chance.append(len(overlap) / len(edges))
    return float(np.mean(hits)), float(np.mean(chance))


def evaluate(sp, *, loc=True, **config):
    """Every number the guide reports for one configuration on one split."""
    cfg = {"n_bands": N_BANDS, "topology": TOPOLOGY, "scale": SCALE_NAME, "mode": "equal",
           "regime": REGIME, "n_components": N_COMPONENTS, "bin_factor": 1, **config}
    fm = fit_model(sp, **cfg)
    ix, qd, prep = sp["ix"], sp["qc_def"], fm["prep"]
    y_qc = y_all[ix["qc"]]
    t0 = time.perf_counter()
    T = fm["triage"].assess(prep(qd.X), y_qc).table
    t_audit = time.perf_counter() - t0
    c = T["conflict"].to_numpy()
    fam = {k: roc_auc_score(qd.degraded[(qd.kind == k) | (qd.degraded == 0)],
                            c[(qd.kind == k) | (qd.degraded == 0)]) for k in degrade.PAPER_FAMILIES}
    row = {"repeat": sp["repeat"], **cfg,
           "channels_per_band": int(np.median([b.n_features for b in fm["bands"].bands])),
           "conflict_vs_degraded": roc_auc_score(qd.degraded, c),
           "conflict_vs_wrong": roc_auc_score(1 - T["correct"], c),
           "local": fam["local contamination"],
           "global_mean": float(np.mean([fam[k] for k in degrade.PAPER_FAMILIES if k != "local contamination"])),
           "acc_in_cohort": accuracy_score(y_all[ix["ref"]], fm["model"].predict(prep(MM_all[ix["ref"]]))),
           "acc_external": accuracy_score(y_B, fm["model"].predict(prep(MM_B))),
           "t_fit_s": fm["t_fit"], "t_audit_s": t_audit,
           "orientation_margin": fm["triage"].orientation_margin_}
    if loc:
        row["loc_hit"], row["loc_chance"] = localise(sp, fm)
    return row


In [2]:
def check(rows):
    """Compare this run with the values quoted in the manuscript and the response letter."""
    t = pd.DataFrame(rows, columns=["quantity", "this_run", "quoted", "tol"])
    t["ok"] = (t["this_run"] - t["quoted"]).abs() <= t["tol"]
    print(t.round(4).to_string(index=False))
    print("\nALL QUOTED VALUES REPRODUCE" if t["ok"].all() else
          f"\n{(~t['ok']).sum()} VALUES DO NOT REPRODUCE: correct the text or the run")
    return t

## 1 · Five splits, two tasks

In [3]:
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor


def increment(lab, base_cols, add_col):
    """Held-out AUROC of a logistic model on base_cols, then with add_col added."""
    j1, j2 = train_test_split(np.arange(len(lab)), test_size=0.5, stratify=lab, random_state=SEED + 1)
    z = lambda v: (v - v[j1].mean()) / (v[j1].std() + 1e-12)  # noqa: E731
    out = []
    for cols in (base_cols, base_cols + [add_col]):
        M = np.column_stack([z(np.asarray(c, float)) for c in cols])
        lr = LogisticRegression(max_iter=2000).fit(M[j1], lab[j1])
        out.append(roc_auc_score(lab[j2], lr.predict_proba(M[j2])[:, 1]))
    return out


single, gains = [], []
for rep in range(5):
    sp = make_split(rep)
    fm = fit_model(sp)
    tri, prep, ix = fm["triage"], fm["prep"], sp["ix"]
    Xfit = prep(MM_all[ix["fit"]])
    pca = PCA(n_components=10, svd_solver="full").fit(Xfit)
    lam = pca.explained_variance_
    t2 = lambda X: (pca.transform(X) ** 2 / lam).sum(1)  # noqa: E731
    qr = lambda X: ((X - pca.inverse_transform(pca.transform(X))) ** 2).sum(1)  # noqa: E731
    iso = IsolationForest(n_estimators=300, random_state=SEED).fit(Xfit)
    lof = LocalOutlierFactor(n_neighbors=20, novelty=True).fit(Xfit)
    detectors = {"PCA Hotelling T2": t2, "PCA Q residual": qr,
                 "Isolation forest": lambda X: -iso.score_samples(X),
                 "Local outlier factor": lambda X: -lof.score_samples(X)}

    yq, qd = y_all[ix["qc"]], sp["qc_def"]
    X18, X19 = prep(MM_all[ix["qc"]]), prep(MM_B)
    Xd = prep(qd.X)
    Td = tri.assess(Xd, yq).table
    T18, T19 = tri.assess(X18, yq).table, tri.assess(X19, y_B).table
    tasks = {
        "planted defects": (qd.degraded.astype(int), Xd, Td),
        "2018 -> 2019 change": (np.r_[np.zeros(len(X18)), np.ones(len(X19))].astype(int),
                                np.vstack([X18, X19]), pd.concat([T18, T19], ignore_index=True)),
    }
    for task, (lab, X, T) in tasks.items():
        r = {"split": rep, "task": task,
             "conflict": roc_auc_score(lab, T["conflict"]),
             "1 - confidence": roc_auc_score(lab, 1 - T["confidence"]),
             "pypasi novelty (T2 + Q)": roc_auc_score(lab, T["novelty"])}
        for name, f in detectors.items():
            r[name] = roc_auc_score(lab, f(X))
        single.append(r)
        a, b = increment(lab, [np.log(t2(X)), np.log(qr(X)), 1 - T["confidence"].to_numpy()],
                         T["conflict"].to_numpy())
        gains.append({"split": rep, "task": task, "T2 + Q + confidence": a,
                      "+ conflict": b, "gain_pp": 100 * (b - a)})
    print(f"split {rep} done", flush=True)

single, gains = pd.DataFrame(single), pd.DataFrame(gains)
def summary(df):
    g = df.drop(columns="split").groupby("task", sort=False)
    return (g.mean().round(3).astype(str) + " ± " + g.std().round(3).astype(str)).T
print("Each score on its own (AUROC, mean ± sd over five splits)")
print(summary(single).to_string())
print("\nWhat conflict adds to T2 + Q + confidence (held-out AUROC)")
print(summary(gains).to_string())
print("\ngain positive in", gains.groupby("task", sort=False)["gain_pp"].apply(lambda s: f"{(s > 0).sum()}/5").to_dict())

split 0 done


split 1 done


split 2 done


split 3 done


split 4 done


Each score on its own (AUROC, mean ± sd over five splits)
task                    planted defects 2018 -> 2019 change
conflict                   0.76 ± 0.028       0.749 ± 0.053
1 - confidence             0.45 ± 0.018       0.501 ± 0.065
pypasi novelty (T2 + Q)   0.455 ± 0.025       0.412 ± 0.071
PCA Hotelling T2          0.642 ± 0.016       0.726 ± 0.066
PCA Q residual            0.421 ± 0.009       0.371 ± 0.047
Isolation forest          0.432 ± 0.008        0.39 ± 0.049
Local outlier factor       0.435 ± 0.01       0.434 ± 0.083

What conflict adds to T2 + Q + confidence (held-out AUROC)
task                planted defects 2018 -> 2019 change
T2 + Q + confidence    0.668 ± 0.02       0.794 ± 0.051
+ conflict            0.797 ± 0.038       0.844 ± 0.036
gain_pp              12.877 ± 3.131       4.945 ± 4.813

gain positive in {'planted defects': '5/5', '2018 -> 2019 change': '5/5'}


## 2 · Every value quoted, checked

In [4]:
S = single.groupby("task").mean(numeric_only=True)
G = gains.groupby("task").mean(numeric_only=True)
P, R = "planted defects", "2018 -> 2019 change"
claims = check([
    ("planted: conflict",                 S.loc[P, "conflict"], 0.760, 0.005),
    ("planted: PCA T2",                   S.loc[P, "PCA Hotelling T2"], 0.642, 0.005),
    ("planted: PCA Q",                    S.loc[P, "PCA Q residual"], 0.421, 0.005),
    ("planted: isolation forest",         S.loc[P, "Isolation forest"], 0.432, 0.005),
    ("planted: LOF",                      S.loc[P, "Local outlier factor"], 0.435, 0.005),
    ("planted: pypasi novelty",           S.loc[P, "pypasi novelty (T2 + Q)"], 0.455, 0.005),
    ("change: conflict",                  S.loc[R, "conflict"], 0.749, 0.005),
    ("change: PCA T2",                    S.loc[R, "PCA Hotelling T2"], 0.726, 0.005),
    ("change: PCA Q",                     S.loc[R, "PCA Q residual"], 0.371, 0.005),
    ("change: isolation forest",          S.loc[R, "Isolation forest"], 0.390, 0.005),
    ("change: LOF",                       S.loc[R, "Local outlier factor"], 0.434, 0.005),
    ("planted: T2+Q+confidence",          G.loc[P, "T2 + Q + confidence"], 0.668, 0.005),
    ("planted: + conflict",               G.loc[P, "+ conflict"], 0.797, 0.005),
    ("planted: gain (pp)",                G.loc[P, "gain_pp"], 12.9, 0.15),
    ("change: T2+Q+confidence",           G.loc[R, "T2 + Q + confidence"], 0.794, 0.005),
    ("change: + conflict",                G.loc[R, "+ conflict"], 0.844, 0.005),
    ("change: gain (pp)",                 G.loc[R, "gain_pp"], 4.9, 0.15),
    ("gain positive, planted (of 5)",     float((gains[gains.task == P]["gain_pp"] > 0).sum()), 5, 0),
    ("gain positive, change (of 5)",      float((gains[gains.task == R]["gain_pp"] > 0).sum()), 5, 0),
])
single.to_csv("outlier_comparison.csv", index=False); gains.to_csv("outlier_increment.csv", index=False)

                     quantity  this_run  quoted   tol   ok
            planted: conflict    0.7596   0.760 0.005 True
              planted: PCA T2    0.6416   0.642 0.005 True
               planted: PCA Q    0.4214   0.421 0.005 True
    planted: isolation forest    0.4316   0.432 0.005 True
                 planted: LOF    0.4351   0.435 0.005 True
      planted: pypasi novelty    0.4554   0.455 0.005 True
             change: conflict    0.7489   0.749 0.005 True
               change: PCA T2    0.7260   0.726 0.005 True
                change: PCA Q    0.3710   0.371 0.005 True
     change: isolation forest    0.3903   0.390 0.005 True
                  change: LOF    0.4339   0.434 0.005 True
     planted: T2+Q+confidence    0.6679   0.668 0.005 True
          planted: + conflict    0.7967   0.797 0.005 True
           planted: gain (pp)   12.8770  12.900 0.150 True
      change: T2+Q+confidence    0.7944   0.794 0.005 True
           change: + conflict    0.8438   0.844 0.005 Tr